# 2.1. Implementar un chatbot básico usando reglas para responder preguntas frecuentes

Partiendo de las nociones teóricas y prácticas vistas en la unidad, el alumno va a diseñar y desarrollar un chatbot que pueda responder preguntas frecuentes. Tanto el diseño como el código deben ser entregados describiendo detalladamente cada uno de los procesos involucrados, así como la justificación y pertinencia de su implementación.

# Diseño del chatbot

## ¿Qué es un chatbot basado en reglas?

Un chatbot basado en reglas responde a partir de **coincidencias de patrones**, no de un modelo entrenado: cada regla asocia un conjunto de palabras clave con una respuesta fija, y el chatbot responde con la primera regla cuyas palabras clave aparezcan en el mensaje del usuario. Es el mismo principio del ELIZA de Weizenbaum (1966), el primer chatbot de la historia, que ya usaba coincidencia de patrones en el texto de entrada para elegir una respuesta predefinida.

## Área de interés: FAQs de una tienda en línea

Se aplica el chatbot a las preguntas frecuentes de una **tienda en línea**: envíos, devoluciones, pagos, seguimiento de pedidos, descuentos, cancelaciones y horario de atención. Es un dominio típico para este enfoque porque las preguntas de los clientes son repetitivas y las respuestas ya están estandarizadas por la política de la tienda — no hace falta un modelo de lenguaje para resolverlas, basta con reconocer la intención por sus palabras clave.

In [3]:
import re
import sys
import unicodedata
from dataclasses import dataclass

print(f"Python {sys.version}")

Python 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Base de conocimiento

Diez reglas, ordenadas por prioridad de coincidencia (se explica más abajo por qué el orden importa). Si un mensaje no activa ninguna, el motor usa una respuesta de reserva fija.

| Regla | Palabras clave (ejemplos) | Qué resuelve |
|---|---|---|
| `saludo` | hola, buenos días, buenas tardes | Da la bienvenida y orienta sobre los temas que el chatbot puede resolver |
| `tiempos_envio` | tarda, tiempo de entrega, cuándo llega | Tiempos estimados de entrega nacional e internacional |
| `costo_envio` | costo de envío, envío gratis, cuánto cuesta el envío | Costo del envío y el umbral de envío gratuito |
| `politica_devolucion` | devolución, reembolso, cambio de producto, garantía | Condiciones y plazos para devolver o cambiar un producto |
| `metodos_pago` | métodos de pago, tarjeta, PayPal, transferencia | Formas de pago aceptadas |
| `rastreo_pedido` | rastrear, seguimiento, número de guía | Cómo dar seguimiento a un pedido ya realizado |
| `descuentos` | descuento, cupón, promoción, código promocional | Dónde obtener y cómo aplicar cupones de descuento |
| `cancelar_pedido` | cancelar pedido, cancelar mi orden | Cómo cancelar una compra antes de que se envíe |
| `horario_atencion` | horario, atención al cliente, WhatsApp | Horario y canales de contacto con soporte humano |
| `despedida` | gracias, adiós, hasta luego | Cierra la conversación de forma cordial |

Las palabras clave se escriben ya normalizadas (minúsculas, sin acentos): el motor aplica la misma normalización al mensaje del usuario antes de comparar, para que "¿Cuánto tarda...?" y "cuanto tarda" coincidan igual.

In [4]:
@dataclass
class Regla:
    nombre: str
    palabras_clave: list[str]
    respuesta: str
    descripcion: str


base_conocimiento = [
    Regla(
        nombre="saludo",
        palabras_clave=["hola", "buenos dias", "buenas tardes", "buenas noches", "buenas"],
        respuesta=(
            "¡Hola! Bienvenido/a a nuestra tienda en línea. Puedo ayudarte con dudas sobre "
            "envíos, devoluciones, métodos de pago, seguimiento de pedidos, descuentos o "
            "nuestro horario de atención. ¿En qué te ayudo?"
        ),
        descripcion="Saludo inicial y presentación de los temas disponibles",
    ),
    Regla(
        nombre="tiempos_envio",
        palabras_clave=["tarda", "tiempo de entrega", "tiempo de envio", "cuando llega"],
        respuesta=(
            "Los pedidos dentro del país se entregan en 3 a 5 días hábiles, y los envíos "
            "internacionales en 7 a 12 días hábiles, dependiendo del destino y la paquetería asignada."
        ),
        descripcion="Tiempos estimados de entrega",
    ),
    Regla(
        nombre="costo_envio",
        palabras_clave=["costo de envio", "cuanto cuesta el envio", "envio gratis", "cobran de envio", "precio del envio"],
        respuesta=(
            "El envío es gratuito en compras a partir de $999 MXN. Por debajo de ese monto, el "
            "costo de envío estándar es de $99 MXN, calculado en el carrito según tu código postal."
        ),
        descripcion="Costo del envío y umbral de envío gratuito",
    ),
    Regla(
        nombre="politica_devolucion",
        palabras_clave=["devolucion", "devolver", "reembolso", "cambio de producto", "garantia"],
        respuesta=(
            "Puedes solicitar la devolución o cambio de un producto dentro de los 30 días "
            "naturales posteriores a la entrega, siempre que esté en su empaque original y sin "
            "señales de uso. El reembolso se aplica al método de pago original en 5 a 10 días hábiles."
        ),
        descripcion="Condiciones y plazos de devolución o cambio",
    ),
    Regla(
        nombre="metodos_pago",
        palabras_clave=["metodos de pago", "como pago", "tarjeta", "paypal", "transferencia", "efectivo"],
        respuesta=(
            "Aceptamos tarjetas de crédito y débito (Visa, Mastercard, American Express), PayPal, "
            "transferencia bancaria y pago en efectivo en tiendas de conveniencia afiliadas."
        ),
        descripcion="Formas de pago aceptadas",
    ),
    Regla(
        nombre="rastreo_pedido",
        palabras_clave=["rastrear", "rastreo", "seguimiento", "donde esta mi pedido", "estado de mi pedido", "numero de guia"],
        respuesta=(
            "Puedes rastrear tu pedido con el número de guía que enviamos por correo al confirmarse "
            "el envío, ingresándolo en la sección 'Mis pedidos' de tu cuenta o en el sitio de la paquetería."
        ),
        descripcion="Seguimiento de un pedido ya realizado",
    ),
    Regla(
        nombre="descuentos",
        palabras_clave=["descuento", "cupon", "promocion", "oferta", "codigo promocional"],
        respuesta=(
            "Publicamos cupones y promociones de temporada en nuestro boletín y redes sociales. "
            "Ingresa el código en el campo 'Cupón de descuento' durante el pago, antes de finalizar la compra."
        ),
        descripcion="Dónde obtener y cómo aplicar descuentos",
    ),
    Regla(
        nombre="cancelar_pedido",
        palabras_clave=["cancelar pedido", "cancelar mi pedido", "cancelar mi orden", "cancelar compra"],
        respuesta=(
            "Puedes cancelar tu pedido sin costo siempre que aún no haya sido enviado, desde la "
            "sección 'Mis pedidos'. Si el paquete ya salió del almacén, deberás recibirlo y "
            "solicitar una devolución."
        ),
        descripcion="Cancelación de un pedido antes del envío",
    ),
    Regla(
        nombre="horario_atencion",
        palabras_clave=["horario", "atencion al cliente", "telefono de contacto", "correo de contacto", "whatsapp"],
        respuesta=(
            "Nuestro equipo de atención al cliente está disponible de lunes a viernes de 9:00 a "
            "18:00 h y sábados de 10:00 a 14:00 h, por chat en el sitio, correo o WhatsApp."
        ),
        descripcion="Horario y canales de contacto con soporte humano",
    ),
    Regla(
        nombre="despedida",
        palabras_clave=["gracias", "adios", "hasta luego", "nos vemos"],
        respuesta="¡Gracias por tu visita! Si tienes otra duda, aquí estaré. Que tengas un excelente día.",
        descripcion="Cierre cordial de la conversación",
    ),
]

print(f"Reglas en la base de conocimiento: {len(base_conocimiento)}")

Reglas en la base de conocimiento: 10


## Motor de coincidencia

`normalizar` deja el texto en minúsculas y sin acentos ni signos de puntuación, para que la comparación de palabras clave no falle por mayúsculas o tildes. `responder` recorre `base_conocimiento` **en orden** y devuelve la respuesta de la primera regla cuyas palabras clave aparezcan en el mensaje ya normalizado; si ninguna aplica, usa `RESPUESTA_RESERVA`.

In [5]:
RESPUESTA_RESERVA = (
    "No logré identificar tu pregunta. ¿Podrías reformularla con otras palabras o "
    "prefieres que te comunique con un asesor humano?"
)


def normalizar(texto: str) -> str:
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def responder(pregunta: str, base_reglas: list[Regla]) -> tuple[str, str]:
    texto = normalizar(pregunta)
    for regla in base_reglas:
        if any(palabra_clave in texto for palabra_clave in regla.palabras_clave):
            return regla.nombre, regla.respuesta
    return "sin_coincidencia", RESPUESTA_RESERVA

## Resolución de conflictos: el orden de las reglas importa

Si un mensaje contiene palabras clave de más de una regla, gana la que aparece primero en `base_conocimiento` — el mismo mecanismo de prioridad por orden que se usó en la Práctica 1.1 para las reglas de decisión. Por ejemplo, *"¿Cuánto tarda el envío y puedo pagar con tarjeta?"* contiene tanto `tarda` (regla `tiempos_envio`) como `tarjeta` (regla `metodos_pago`); como `tiempos_envio` está antes en la lista, es la que responde. Esto se comprueba más abajo, en la batería de prueba.

## Demostración con una conversación

Antes de la batería de prueba completa, se simula una mini conversación para ver el motor en acción, mensaje por mensaje.

In [6]:
conversacion_demo = [
    "Hola, buenos días",
    "¿Cuánto tarda en llegar mi pedido?",
    "¿Puedo pagar con PayPal?",
    "Muchas gracias, hasta luego",
]

for mensaje in conversacion_demo:
    regla_disparada, respuesta = responder(mensaje, base_conocimiento)
    print(f"Usuario : {mensaje}")
    print(f"Regla   : {regla_disparada}")
    print(f"Chatbot : {respuesta}\n")

Usuario : Hola, buenos días
Regla   : saludo
Chatbot : ¡Hola! Bienvenido/a a nuestra tienda en línea. Puedo ayudarte con dudas sobre envíos, devoluciones, métodos de pago, seguimiento de pedidos, descuentos o nuestro horario de atención. ¿En qué te ayudo?

Usuario : ¿Cuánto tarda en llegar mi pedido?
Regla   : tiempos_envio
Chatbot : Los pedidos dentro del país se entregan en 3 a 5 días hábiles, y los envíos internacionales en 7 a 12 días hábiles, dependiendo del destino y la paquetería asignada.

Usuario : ¿Puedo pagar con PayPal?
Regla   : metodos_pago
Chatbot : Aceptamos tarjetas de crédito y débito (Visa, Mastercard, American Express), PayPal, transferencia bancaria y pago en efectivo en tiendas de conveniencia afiliadas.

Usuario : Muchas gracias, hasta luego
Regla   : despedida
Chatbot : ¡Gracias por tu visita! Si tienes otra duda, aquí estaré. Que tengas un excelente día.



# Prueba del sistema

La batería cubre las diez reglas al menos una vez, más dos casos especiales: uno de **solape** de palabras clave (para comprobar la prioridad por orden) y uno **sin ninguna coincidencia** (para comprobar la respuesta de reserva).

In [7]:
preguntas_prueba = [
    "Hola, buenas tardes",
    "¿Cuánto tiempo tarda la entrega de mi pedido?",
    "¿Cuánto cuesta el envío a mi domicilio?",
    "Quiero hacer una devolución de un producto",
    "¿Qué métodos de pago aceptan?",
    "¿Cómo puedo rastrear mi pedido?",
    "¿Tienen algún cupón de descuento disponible?",
    "Necesito cancelar mi pedido",
    "¿Cuál es su horario de atención al cliente?",
    "Muchas gracias por la ayuda, adiós",
    "Quiero saber cuánto tarda el envío y si puedo pagar con tarjeta",
    "¿Venden en tiendas físicas también?",
]

print(f"{'Pregunta':<58} | {'Regla disparada'}")
print("-" * 90)
for pregunta in preguntas_prueba:
    regla_disparada, _ = responder(pregunta, base_conocimiento)
    print(f"{pregunta:<58} | {regla_disparada}")

Pregunta                                                   | Regla disparada
------------------------------------------------------------------------------------------
Hola, buenas tardes                                        | saludo
¿Cuánto tiempo tarda la entrega de mi pedido?              | tiempos_envio
¿Cuánto cuesta el envío a mi domicilio?                    | costo_envio
Quiero hacer una devolución de un producto                 | politica_devolucion
¿Qué métodos de pago aceptan?                              | metodos_pago
¿Cómo puedo rastrear mi pedido?                            | rastreo_pedido
¿Tienen algún cupón de descuento disponible?               | descuentos
Necesito cancelar mi pedido                                | cancelar_pedido
¿Cuál es su horario de atención al cliente?                | horario_atencion
Muchas gracias por la ayuda, adiós                         | despedida
Quiero saber cuánto tarda el envío y si puedo pagar con tarjeta | tiempos_envio
¿Venden

## Lectura de resultados

Las primeras diez preguntas disparan cada una la regla que se diseñó para ella, sin sorpresas. Las dos últimas son las que valen la pena revisar:

- **Solape** — *"Quiero saber cuánto tarda el envío y si puedo pagar con tarjeta"* contiene tanto `tarda` como `tarjeta`, palabras clave de `tiempos_envio` y `metodos_pago` respectivamente. El motor responde con `tiempos_envio`, no porque sea "más correcta", sino porque esa regla está antes en `base_conocimiento`: confirma que el orden de la lista es el mecanismo real de resolución de conflictos, igual que en la Práctica 1.1.
- **Sin coincidencia** — *"¿Venden en tiendas físicas también?"* no contiene ninguna palabra clave de las diez reglas, así que cae en `sin_coincidencia` y el chatbot responde con `RESPUESTA_RESERVA` en vez de fallar o inventar una respuesta — el comportamiento esperado ante una pregunta fuera del alcance diseñado.

# Conclusiones

El chatbot implementado resuelve las FAQs de una tienda en línea con un motor deliberadamente simple: una lista de reglas condición-respuesta y una función que recorre esa lista una sola vez por mensaje, sin hechos derivados ni pasadas múltiples como en la Práctica 1.1. Aun así, la batería de prueba mostró que conserva las dos propiedades que hacen útil a cualquier sistema de reglas: es **transparente** (siempre se sabe qué regla respondió y por qué) y es **extensible** — agregar una nueva pregunta frecuente solo requiere añadir un objeto `Regla` más a la lista, sin tocar el motor.

La limitación es la contraparte de esa simplicidad: el chatbot solo reconoce las palabras clave que se le dieron explícitamente, no entiende sinónimos no listados ni mantiene contexto entre mensajes (cada pregunta se evalúa de forma independiente). Para un catálogo de FAQs acotado y conocido de antemano, como el de una tienda en línea, esa limitación es aceptable; para conversaciones abiertas o con lenguaje muy variable haría falta un enfoque estadístico o basado en un modelo de lenguaje.

## Referencias

Weizenbaum, J. (1966). ELIZA — A computer program for the study of natural language communication between man and machine. *Communications of the ACM*, 9(1), 36–45.

Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

Hernández Parada, M. S. (2026). *Maestría en Inteligencia Artificial — Prácticas* [Repositorio de GitHub]. GitHub. [https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas/blob/main/Cuatrimestre%203/7%20-%20Razonamiento%20Inteligente/Practica%202.1/Implementar%20un%20chatbot%20b%C3%A1sico%20usando%20reglas%20para%20responder%20preguntas%20frecuentes.ipynb](https://github.com/RKCbas/Maestria-en-Inteligencia-Artificial---Practicas/blob/main/Cuatrimestre%203/7%20-%20Razonamiento%20Inteligente/Practica%202.1/Implementar%20un%20chatbot%20b%C3%A1sico%20usando%20reglas%20para%20responder%20preguntas%20frecuentes.ipynb)